# T07 · Descarga de la muestra TCIA/IDC y manifest.csv

Descarga **solo la muestra** (`data/sample_ids.csv`, 60 pacientes) de la colección
`fdg_pet_ct_lesions` vía **IDC** (`idc-index`, espejo TCIA *Images - Defaced*, CC BY 4.0),
convierte DICOM → NIfTI con el script oficial de lab-midas, calcula `n_lesions` y
`tmtv_ml`, genera `data/manifest.csv` y empaqueta la muestra para publicarla como
**dataset privado de Kaggle**.

1. Panel derecho *Session options*: *Internet* **activado** (GPU no hace falta).
2. Si el PR aún no está en `main`, cambia `BRANCH` por `feat/t07-download-manifest`.
3. En *Secrets* añade `HF_TOKEN` (token de HF con permiso de escritura) y, si
   quieres, `HF_REPO_ID` (p. ej. `tu-usuario/fdg-pet-ct-lesions-t07`).
4. Ejecuta todas las celdas en orden. La descarga son ~29 GB de DICOM; usa disco
   de ~90 GB.
5. La última celda sube `data/raw/` completo (~30 GB) a un dataset **privado**
   de Hugging Face e imprime el enlace. Sin token, genera un zip local.

FDAT (fuente NIfTI) está caído; ver `docs/reproducibilidad/licencia_datos.md`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Ronald1301/Lymphoma-Machine-Learning.git"
BRANCH = "main"  # antes del merge: "feat/t07-download-manifest"
REPO_DIR = Path("/kaggle/working/Lymphoma-Machine-Learning")

def run(*cmd: str) -> None:
    print("$", *cmd)
    subprocess.run(cmd, check=True)

if REPO_DIR.exists():
    run("git", "-C", str(REPO_DIR), "pull", "--ff-only")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR))

run(
    sys.executable, "-m", "pip", "install", "-q",
    "idc-index==0.13.5",
    "dicom2nifti==2.5.0",
    "pydicom==3.0.1",
    "nibabel==5.5.0",
    "huggingface_hub==2.4.0",
    "pandas", "scipy", "numpy", "nilearn", "tqdm",
)
sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)

## Muestra y consulta IDC

Se piden las series `PT`, `CT` y `SEG`, pero **solo del estudio baseline**:
5 pacientes tienen un segundo estudio de seguimiento en IDC que no debe descargarse.
El sufijo de `study_id` (p. ej. `..._71302`) son los últimos 5 dígitos del
`StudyInstanceUID` de DICOM, así que sirve para elegir el estudio correcto.

In [ ]:
from idc_index import IDCClient
import pandas as pd

sample = pd.read_csv(REPO_DIR / "data" / "sample_ids.csv", dtype=str)
patient_ids = sorted(sample["patient_id"].unique())
print(len(patient_ids), "pacientes en la muestra")

client = IDCClient.client()
id_list = ", ".join(f"'{p}'" for p in patient_ids)
query = f"""
SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    Modality,
    SeriesDescription,
    series_size_MB
FROM index
WHERE collection_id = 'fdg_pet_ct_lesions'
  AND PatientID IN ({id_list})
  AND Modality IN ('PT', 'CT', 'SEG')
"""
all_series = client.sql_query(query)
print(len(all_series), "series en todos los estudios de esos pacientes")

# Quedarse solo con el estudio baseline de sample_ids.csv.
suffix = sample["study_id"].str.rsplit("_", n=1).str[-1]
wanted = pd.DataFrame({
    "PatientID": sample["patient_id"],
    "suffix": suffix,
    "date": sample["date"],
})
all_series["suffix"] = all_series["StudyInstanceUID"].str[-5:]
series = all_series.merge(wanted, on=["PatientID", "suffix"], how="inner")

assert len(wanted) == len(patient_ids), "study_id duplicado en la muestra"
assert series["StudyInstanceUID"].nunique() == len(patient_ids), "un estudio no mappeó"
fecha_idc = series["StudyDate"].astype(str).str.replace("-", "", regex=False)
fecha_sample = series["date"].str.replace("-", "", regex=False)
assert (fecha_idc == fecha_sample).all(), "fecha de estudio distinta"
assert series.groupby("SeriesInstanceUID").size().max() == 1, "serie duplicada"
print(len(series), "series del baseline (60 estudios x 3 modalidades)")
print(series.groupby("Modality").size().to_dict())
print("MB totales:", round(series["series_size_MB"].sum(), 1))

## Descarga de las series del baseline

In [ ]:
DICOM_DIR = Path("/kaggle/working/dicom")
DICOM_DIR.mkdir(parents=True, exist_ok=True)

uids = series["SeriesInstanceUID"].tolist()
print("descargando", len(uids), "series a", DICOM_DIR)
# Sin el nivel collection_id: el conversor espera paciente/estudio/serie.
client.download_dicom_series(
    seriesInstanceUID=uids,
    downloadDir=str(DICOM_DIR),
    dirTemplate="%PatientID/%StudyInstanceUID/%Modality_%SeriesInstanceUID",
)
print("listo")

## Conversión DICOM → NIfTI (script oficial lab-midas)

Genera `SUV.nii.gz`, `CTres.nii.gz`, `PET.nii.gz`, `CT.nii.gz` y `SEG.nii.gz`
en `data/raw/<paciente>/<estudio>/`, con la estructura que espera el contrato.

In [ ]:
TCIA_SCRIPTS = Path("/kaggle/working/TCIA_processing")
if not TCIA_SCRIPTS.exists():
    run("git", "clone", "--depth", "1", "--branch", "master",
        "https://github.com/lab-midas/TCIA_processing.git", str(TCIA_SCRIPTS))

RAW_DIR = REPO_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

run(
    sys.executable,
    str(TCIA_SCRIPTS / "tcia_dicom_to_nifti.py"),
    str(DICOM_DIR),
    str(RAW_DIR),
)

## Validar estructura de salida y mapear a la muestra

In [ ]:
patients_out = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())
print(len(patients_out), "pacientes convertidos")
missing = set(patient_ids) - set(patients_out)
print("faltantes:", len(missing), sorted(missing)[:10])

convertidos = 0
sin_mascara = []
for pid in patient_ids:
    for study_dir in (RAW_DIR / pid).glob("*"):
        if (study_dir / "SEG.nii.gz").is_file():
            convertidos += 1
        else:
            sin_mascara.append(str(study_dir))
print("estudios con máscara:", convertidos, "| sin máscara:", len(sin_mascara))

example = next(RAW_DIR.glob("*/*/"), None)
if example:
    print("estructura ejemplo:", example)
    print(sorted(f.name for f in example.glob("*.nii.gz")))

## Construir `manifest.csv` con métricas de la máscara

In [ ]:
from lymphoma.data.manifest import build_manifest, write_manifest

OUT_DIR = Path("/kaggle/working/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = build_manifest(RAW_DIR, sample, repo_root=REPO_DIR)
write_manifest(manifest, OUT_DIR / "manifest.csv")
write_manifest(manifest, REPO_DIR / "data" / "manifest.csv")
print(manifest[["study_id", "diagnosis", "n_lesions", "tmtv_ml"]].head(10))
print(
    "filas:", len(manifest),
    "con máscara:", int((manifest["mask_path"] != "").sum()),
)

## Compartir la muestra con el equipo (Hugging Face)

Se sube `data/raw/` completo: los 5 NIfTI por estudio (~30 GB). Se usa
`upload_folder` porque el navegador de HF limita a 5 GB por archivo y un zip
de ~30 GB no pasaría; cada NIfTI pesa ~100–300 MB. Después hay que añadir al
equipo en *Settings → Members* del repo.

In [ ]:
import os
import shutil
from huggingface_hub import HfApi, upload_folder

def read_secret(name: str) -> str:
    """Kaggle expone los secrets por UserSecretsClient (o por variable de entorno)."""
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name).strip()
    except Exception as exc:
        print(f"No se pudo leer {name}: {exc}")
        return ""

HF_REPO_ID = read_secret("HF_REPO_ID")
HF_TOKEN = read_secret("HF_TOKEN")

shutil.copy(REPO_DIR / "data" / "sample_ids.csv", OUT_DIR / "sample_ids.csv")
print("Salidas locales en", OUT_DIR)

if HF_TOKEN and HF_REPO_ID:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(
        repo_id=HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True
    )
    upload_folder(
        folder_path=str(RAW_DIR),
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        path_in_repo="raw",
        token=HF_TOKEN,
    )
    api.upload_file(
        path_or_fileobj=str(OUT_DIR / "manifest.csv"),
        path_in_repo="manifest.csv",
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    api.upload_file(
        path_or_fileobj=str(OUT_DIR / "sample_ids.csv"),
        path_in_repo="sample_ids.csv",
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    url = f"https://huggingface.co/datasets/{HF_REPO_ID}"
    print("Subido a", url)
    print("Siguiente paso: pegar ese enlace en el issue #7.")
else:
    print("Sin HF_TOKEN o HF_REPO_ID: no se subió nada a Hugging Face.")
    print("Añade esos Secrets y vuelve a ejecutar esta celda.")
    print("Como alternativa se genera un zip local:")
    if DICOM_DIR.exists():
        shutil.rmtree(DICOM_DIR)  # liberar ~29 GB antes de comprimir
    zip_path = OUT_DIR / "fdg_sample_t07.zip"
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", RAW_DIR)
    print("zip:", zip_path, round(zip_path.stat().st_size / 1e9, 2), "GB")